In [1]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import suncalc

import dask.dataframe as dd
from pathlib import Path
from tqdm import tqdm
import re
import pytz

import datetime as dt

import matplotlib.pyplot as plt
from matplotlib import colors
import soundfile as sf
import matplotlib.patches as patches

In [2]:
import sys

sys.path.append("../src")
sys.path.append("../src/activity")

In [3]:
import subsampling as ss
import activity_assembly as actvt
import bout.assembly as bt
import comparison.data_assembly as comp
import comparison.plot as complot
from core import SITE_NAMES

from cli import get_file_paths
import plot
import pipeline

In [4]:
avail = np.arange(0, 180, 2) + 2
reset_3 = avail[np.where((3*60 % avail) == 0)[0]]
reset_4 = avail[np.where((4*60 % avail) == 0)[0]]
reset_6 = avail[np.where((6*60 % avail) == 0)[0]]
reset_12 = avail[np.where((12*60 % avail) == 0)[0]]
reset_24 = avail[np.where((24*60 % avail) == 0)[0]]

In [5]:
dt_starts = {'Carp high':dt.datetime(2022, 7, 15, 3, 0, 0),
           'Telephone high':dt.datetime(2022, 7, 15, 3, 0, 0)}
dt_ends = {'Carp high':dt.datetime(2022, 10, 17, 13, 0, 0),
           'Telephone high':dt.datetime(2022, 10, 17, 13, 0, 0)}
high_activity_types = {'Carp':'LF', 'Telephone':'HF'}

In [6]:
step = 1/10
step_by = np.arange(step, 1, step)
# step_by = step_by[step_by>=(1/6)]
step_by

array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

In [7]:
step = 1/6
step_by = np.arange(0, (3/6)+step, step)
step_by = step_by[step_by>=(1/6)]
step_by

array([0.16666667, 0.33333333, 0.5       ])

In [8]:
cycle_lengths = [6, 10, 30]
percent_ons = step_by
data_params = dict()
data_params["cycle_lengths"] = cycle_lengths
data_params["percent_ons"] = percent_ons
dc_tags = ss.get_list_of_dc_tags(cycle_lengths, percent_ons)
dc_tags

['30of30',
 '1of6',
 '2of6',
 '3of6',
 '2of10',
 '3of10',
 '5of10',
 '5of30',
 '10of30',
 '15of30']

In [9]:
site_keys = ['Carp']
type_keys = ['LF']
data_params["dc_tags"] = dc_tags
data_params['cur_dc_tag'] = '30of30'
data_params['recording_start'] = '00:00'
data_params['recording_end'] = '16:00'
data_params['assembly_type'] = 'kmeans'
site_key = site_keys[0]
type_key = type_keys[0]

type_key = high_activity_types[site_key]
actvt_key = 'high'
data_params['start'] = dt_starts[f'{site_key} {actvt_key}']
data_params['end'] = dt_ends[f'{site_key} {actvt_key}']
data_params['P'] = 0.4

In [10]:
def add_noise_to_group(group, frac):
    noise_df = pd.DataFrame()
    noise_start_times = np.random.uniform(0, 1800, np.floor(frac*group.shape[0]).astype(int))
    noise_df['start_time'] = noise_start_times
    noise_df['end_time'] = noise_start_times + 0.01
    noise_df['low_freq'] = [24000]*noise_df.shape[0]
    noise_df['high_freq'] = [80000]*noise_df.shape[0]
    noise_df['class'] = ['NOISE']*noise_df.shape[0]

    added_noise_file_dets = pd.concat([group, noise_df])
    if frac == 0:
        assert(len(noise_df)==0)
        assert (added_noise_file_dets['call_start_time'].equals(group['call_start_time']))
    file_dts = pd.to_datetime(added_noise_file_dets['input_file'], format='%Y%m%d_%H%M%S', exact=False).ffill().bfill()

    anchor_start_times = file_dts + pd.to_timedelta(added_noise_file_dets['start_time'].values.astype('float64'), unit='S')
    anchor_end_times = file_dts + pd.to_timedelta(added_noise_file_dets['end_time'].values.astype('float64'), unit='S') 
    added_noise_file_dets['call_end_time'] = anchor_end_times
    added_noise_file_dets['call_start_time'] = anchor_start_times
    added_noise_file_dets['ref_time'] = anchor_start_times
    added_noise_file_dets = added_noise_file_dets.sort_values(by='call_start_time')

    return added_noise_file_dets

def generate_activity_btp_for_false_positives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=False):
    btp_mod_columns = pd.DataFrame()
    location_df = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)
    bout_params = bt.get_bout_params_from_location(location_df, data_params)

    metric_col_name = f'{data_params["metric_tag"]} ({dc_tag})'
    btp_cont_column = comp.get_continuous_btp_partitioned_for_dc_scheme(metric_col_name, location_df.copy(), data_params, bout_params)
    fractions = np.arange(0.0, 0.51, 0.05)
    for i in tqdm(range(len(fractions))):
        frac = fractions[i]
        data_params['cur_dc_tag'] = dc_tag
        cycle_length_in_mins = int(data_params['cur_dc_tag'].split('of')[1])
        time_on_in_mins = int(data_params['cur_dc_tag'].split('of')[0])
        time_on_in_secs = (60*time_on_in_mins)

        added_noise_location_df = location_df.groupby(by='input_file_dt', group_keys=False).apply(lambda x : add_noise_to_group(x, frac))
        dc_applied_df = ss.simulate_dutycycle_on_detections(added_noise_location_df.copy(), data_params)
        dc_applied_df['freq_group'] = [data_params['type_tag']]*dc_applied_df.shape[0]
        # comp.does_duty_cycled_df_have_less_dets_than_original(dc_applied_df, location_df)
        bout_metrics = bt.generate_bout_metrics_for_location_and_freq(dc_applied_df, data_params, bout_params)
        bout_duration = actvt.get_bout_duration_per_cycle(bout_metrics, cycle_length_in_mins)
        bout_time_percentage = actvt.get_btp_per_time_on(bout_duration, time_on_in_secs)
        data_params['cur_dc_tag'] = f'{round(frac, 2)}'
        bout_time_percentage_modified = actvt.filter_and_prepare_metric(bout_time_percentage, data_params)
        bout_time_percentage_modified = bout_time_percentage_modified.set_index("datetime_UTC")
        ss.are_there_expected_number_of_cycles(dc_applied_df, bout_time_percentage_modified, cycle_length_in_mins, data_params)

        btp_mod_columns = pd.concat([btp_mod_columns, bout_time_percentage_modified], axis=1)

    if save:
        btp_mod_columns.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fp_error_mod_btp_TYPE_SITE_summary"]}.csv')
        btp_cont_column.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fp_error_base_btp_TYPE_SITE_summary"]}.csv')

    return btp_mod_columns, btp_cont_column


def generate_activity_call_rate_for_false_positives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=False):
    activity_arr = pd.DataFrame()
    location_df = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)

    metric_col_name = f'{data_params["metric_tag"]} ({dc_tag})'
    callrate_arr = comp.get_continuous_call_rates_partitioned_for_dc_scheme(metric_col_name, file_paths, data_params)
    fractions = np.arange(0.0, 0.51, 0.05)
    for i in tqdm(range(len(fractions))):
        frac = fractions[i]
        data_params['cur_dc_tag'] = dc_tag
        cycle_length_in_mins = int(data_params['cur_dc_tag'].split('of')[1])
        time_on_in_mins = int(data_params['cur_dc_tag'].split('of')[0])

        added_noise_location_df = location_df.groupby(by='input_file_dt', group_keys=False).apply(lambda x : add_noise_to_group(x, frac))
        dc_applied_df = ss.simulate_dutycycle_on_detections(added_noise_location_df.copy(), data_params)
        dc_applied_df['freq_group'] = [data_params['type_tag']]*dc_applied_df.shape[0]
        # comp.does_duty_cycled_df_have_less_dets_than_original(dc_applied_df, location_df)
        num_of_detections = actvt.get_number_of_detections_per_cycle(dc_applied_df, cycle_length_in_mins)        
        call_rate = actvt.get_metric_per_time_on(num_of_detections, time_on_in_mins)
        data_params['cur_dc_tag'] = f'{round(frac, 2)}'
        call_rate_dc_column = actvt.filter_and_prepare_metric(call_rate, data_params)
        call_rate_dc_column = call_rate_dc_column.set_index("datetime_UTC")
        ss.are_there_expected_number_of_cycles(dc_applied_df, call_rate_dc_column, cycle_length_in_mins, data_params)
        
        activity_arr = pd.concat([activity_arr, call_rate_dc_column], axis=1)

    if save:
        activity_arr.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fp_error_mod_callrate_TYPE_SITE_summary"]}.csv')
        callrate_arr.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fp_error_base_callrate_TYPE_SITE_summary"]}.csv')

    return activity_arr, callrate_arr


def generate_activity_index_percent_for_false_positives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=False):
    activity_arr = pd.DataFrame()
    location_df = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)

    metric_col_name = f'{data_params["metric_tag"]} ({dc_tag})'
    actvtind_cont_column = comp.get_continuous_activity_index_partitioned_for_dc_scheme(metric_col_name, file_paths, data_params)
    fractions = np.arange(0.0, 0.51, 0.05)
    for i in tqdm(range(len(fractions))):
        frac = fractions[i]
        data_params['cur_dc_tag'] = dc_tag
        cycle_length_in_mins = int(data_params['cur_dc_tag'].split('of')[1])
        data_params['time_on_in_secs'] = int(data_params['cur_dc_tag'].split('of')[0]) * 60

        added_noise_location_df = location_df.groupby(by='input_file_dt', group_keys=False).apply(lambda x : add_noise_to_group(x, frac))
        dc_applied_df = ss.simulate_dutycycle_on_detections(added_noise_location_df.copy(), data_params)
        dc_applied_df['freq_group'] = [data_params['type_tag']]*dc_applied_df.shape[0]
        # comp.does_duty_cycled_df_have_less_dets_than_original(dc_applied_df, location_df)
        num_blocks_of_presence = actvt.get_activity_index_per_cycle(dc_applied_df, data_params)        
        activity_ind_percent = actvt.get_activity_index_per_time_on_index(num_blocks_of_presence, data_params)
        data_params['cur_dc_tag'] = f'{round(frac, 2)}'
        ind_percent_dc_column = actvt.filter_and_prepare_metric(activity_ind_percent, data_params)
        ind_percent_dc_column = ind_percent_dc_column.set_index("datetime_UTC")
        ss.are_there_expected_number_of_cycles(dc_applied_df, ind_percent_dc_column, cycle_length_in_mins, data_params)
        
        activity_arr = pd.concat([activity_arr, ind_percent_dc_column], axis=1)

    if save:
        activity_arr.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fp_error_mod_actind_TYPE_SITE_summary"]}.csv')
        actvtind_cont_column.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fp_error_base_actind_TYPE_SITE_summary"]}.csv')

    return activity_arr, actvtind_cont_column

In [11]:
for site_key in site_keys:
    for type_key in type_keys:
        for dc_tag in dc_tags:
            print(site_key, type_key, dc_tag)
            data_params['metric_tag'] = 'bout_time_percentage'
            data_params["site_tag"] = site_key
            data_params["site_name"] = SITE_NAMES[site_key]
            data_params["type_tag"] = type_key
            data_params["detector_tag"] = 'bd2'
            file_paths = get_file_paths(data_params)

            activitybout_arr_fp, btp_arr_fp = generate_activity_btp_for_false_positives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=True)

Carp LF 30of30


100%|██████████| 11/11 [03:06<00:00, 16.99s/it]


Carp LF 1of6


100%|██████████| 11/11 [01:45<00:00,  9.55s/it]


Carp LF 2of6


...

100%|██████████| 11/11 [01:18<00:00,  7.11s/it]


Carp LF 10of30


100%|██████████| 11/11 [01:40<00:00,  9.11s/it]


Carp LF 15of30


100%|██████████| 11/11 [01:58<00:00, 10.81s/it]


In [12]:
for site_key in site_keys:
    for type_key in type_keys:
        for dc_tag in dc_tags:
            print(site_key, type_key, dc_tag)
            data_params['metric_tag'] = 'call_rate'
            data_params["site_tag"] = site_key
            data_params["site_name"] = SITE_NAMES[site_key]
            data_params["type_tag"] = type_key
            data_params["detector_tag"] = 'bd2'
            file_paths = get_file_paths(data_params)

            activitycallrate_arr_fp, callrate_arr_fp = generate_activity_call_rate_for_false_positives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=True)

Carp LF 30of30


100%|██████████| 11/11 [01:19<00:00,  7.26s/it]


Carp LF 1of6


100%|██████████| 11/11 [01:16<00:00,  6.98s/it]


Carp LF 2of6


...

100%|██████████| 11/11 [00:52<00:00,  4.80s/it]


Carp LF 10of30


100%|██████████| 11/11 [00:54<00:00,  4.95s/it]


Carp LF 15of30


100%|██████████| 11/11 [00:53<00:00,  4.90s/it]


In [13]:
for site_key in site_keys:
    for type_key in type_keys:
        for dc_tag in dc_tags:
            print(site_key, type_key, dc_tag)
            data_params['metric_tag'] = 'activity_index'
            data_params["site_tag"] = site_key
            data_params["site_name"] = SITE_NAMES[site_key]
            data_params["type_tag"] = type_key
            data_params["detector_tag"] = 'bd2'
            data_params['index_time_block_in_secs'] = 5
            file_paths = get_file_paths(data_params)

            activityind_arr_fp, actvtind_arr_fp = generate_activity_index_percent_for_false_positives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=True)

Carp LF 30of30


100%|██████████| 11/11 [00:58<00:00,  5.32s/it]


Carp LF 1of6


100%|██████████| 11/11 [01:05<00:00,  5.94s/it]


Carp LF 2of6


...

100%|██████████| 11/11 [00:54<00:00,  4.97s/it]


Carp LF 10of30


100%|██████████| 11/11 [00:58<00:00,  5.27s/it]


Carp LF 15of30


100%|██████████| 11/11 [00:57<00:00,  5.22s/it]


In [14]:
def removed_calls_from_group(group, frac):
    group_reduced = group.sample(frac=frac).sort_values(by='call_start_time')
    if frac == 0:
        assert (group_reduced.equals(group))
    return group_reduced

def generate_activity_btp_for_false_negatives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=False):
    btp_mod_columns = pd.DataFrame()
    location_df = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)
    bout_params = bt.get_bout_params_from_location(location_df, data_params)

    metric_col_name = f'{data_params["metric_tag"]} ({dc_tag})'
    btp_cont_column = comp.get_continuous_btp_partitioned_for_dc_scheme(metric_col_name, location_df.copy(), data_params, bout_params)
    fractions = np.arange(1.0, 0.49, -0.05)
    for i in tqdm(range(len(fractions))):
        frac = fractions[i]
        data_params['cur_dc_tag'] = dc_tag
        cycle_length_in_mins = int(data_params['cur_dc_tag'].split('of')[1])
        time_on_in_mins = int(data_params['cur_dc_tag'].split('of')[0])
        time_on_in_secs = (60*time_on_in_mins)

        removed_calls_location_df = location_df.groupby(by='input_file_dt', group_keys=False).apply(lambda x : removed_calls_from_group(x, frac))
        dc_applied_df_reduced = ss.simulate_dutycycle_on_detections(removed_calls_location_df.copy(), data_params)
        # comp.does_duty_cycled_df_have_less_dets_than_original(dc_applied_df_reduced, location_df)
        bout_metrics = bt.generate_bout_metrics_for_location_and_freq(dc_applied_df_reduced, data_params, bout_params)
        bout_duration = actvt.get_bout_duration_per_cycle(bout_metrics, cycle_length_in_mins)
        bout_time_percentage = actvt.get_btp_per_time_on(bout_duration, time_on_in_secs)
        data_params['cur_dc_tag'] = f'{round(frac, 2)}'
        bout_time_percentage_dc_column = actvt.filter_and_prepare_metric(bout_time_percentage, data_params)
        bout_time_percentage_dc_column = bout_time_percentage_dc_column.set_index("datetime_UTC")
        ss.are_there_expected_number_of_cycles(dc_applied_df_reduced, bout_time_percentage_dc_column, cycle_length_in_mins, data_params)

        btp_mod_columns = pd.concat([btp_mod_columns, bout_time_percentage_dc_column], axis=1)

    if save:
        btp_mod_columns.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fn_error_mod_btp_TYPE_SITE_summary"]}.csv')
        btp_cont_column.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fn_error_base_btp_TYPE_SITE_summary"]}.csv')

    return btp_mod_columns, btp_cont_column


def generate_activity_call_rate_for_false_negatives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=False):
    activity_arr = pd.DataFrame()
    location_df = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)

    metric_col_name = f'{data_params["metric_tag"]} ({dc_tag})'
    callrate_arr = comp.get_continuous_call_rates_partitioned_for_dc_scheme(metric_col_name, file_paths, data_params)
    fractions = np.arange(1.0, 0.49, -0.05)
    for i in tqdm(range(len(fractions))):
        frac = fractions[i]
        data_params['cur_dc_tag'] = dc_tag
        cycle_length_in_mins = int(data_params['cur_dc_tag'].split('of')[1])
        time_on_in_mins = int(data_params['cur_dc_tag'].split('of')[0])

        removed_calls_location_df = location_df.groupby(by='input_file_dt', group_keys=False).apply(lambda x : removed_calls_from_group(x, frac))
        dc_applied_df_reduced = ss.simulate_dutycycle_on_detections(removed_calls_location_df.copy(), data_params)
        # comp.does_duty_cycled_df_have_less_dets_than_original(dc_applied_df_reduced, location_df)
        num_of_detections = actvt.get_number_of_detections_per_cycle(dc_applied_df_reduced, cycle_length_in_mins)        
        call_rate = actvt.get_metric_per_time_on(num_of_detections, time_on_in_mins)
        data_params['cur_dc_tag'] = f'{round(frac, 2)}'
        call_rate_dc_column = actvt.filter_and_prepare_metric(call_rate, data_params)
        call_rate_dc_column = call_rate_dc_column.set_index("datetime_UTC")
        ss.are_there_expected_number_of_cycles(dc_applied_df_reduced, call_rate_dc_column, cycle_length_in_mins, data_params)
        
        activity_arr = pd.concat([activity_arr, call_rate_dc_column], axis=1)

    if save:
        activity_arr.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fn_error_mod_callrate_TYPE_SITE_summary"]}.csv')
        callrate_arr.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fn_error_base_callrate_TYPE_SITE_summary"]}.csv')

    return activity_arr, callrate_arr


def generate_activity_index_percent_for_false_negatives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=False):
    activity_arr = pd.DataFrame()
    location_df = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)

    metric_col_name = f'{data_params["metric_tag"]} ({dc_tag})'
    actvtind_cont_column = comp.get_continuous_activity_index_partitioned_for_dc_scheme(metric_col_name, file_paths, data_params)
    fractions = np.arange(1.0, 0.49, -0.05)
    for i in tqdm(range(len(fractions))):
        frac = fractions[i]
        data_params['cur_dc_tag'] = dc_tag
        cycle_length_in_mins = int(data_params['cur_dc_tag'].split('of')[1])
        data_params['time_on_in_secs'] = int(data_params['cur_dc_tag'].split('of')[0]) * 60

        removed_calls_location_df = location_df.groupby(by='input_file_dt', group_keys=False).apply(lambda x : removed_calls_from_group(x, frac))
        dc_applied_df_reduced = ss.simulate_dutycycle_on_detections(removed_calls_location_df.copy(), data_params)
        # comp.does_duty_cycled_df_have_less_dets_than_original(dc_applied_df_reduced, location_df)
        num_blocks_of_presence = actvt.get_activity_index_per_cycle(dc_applied_df_reduced, data_params)        
        activity_ind_percent = actvt.get_activity_index_per_time_on_index(num_blocks_of_presence, data_params)
        data_params['cur_dc_tag'] = f'{round(frac, 2)}'
        ind_percent_dc_column = actvt.filter_and_prepare_metric(activity_ind_percent, data_params)
        ind_percent_dc_column = ind_percent_dc_column.set_index("datetime_UTC")
        ss.are_there_expected_number_of_cycles(dc_applied_df_reduced, ind_percent_dc_column, cycle_length_in_mins, data_params)
        
        activity_arr = pd.concat([activity_arr, ind_percent_dc_column], axis=1)

    if save:
        activity_arr.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fn_error_mod_actind_TYPE_SITE_summary"]}.csv')
        actvtind_cont_column.to_csv(f'{file_paths["duty_cycled_folder"]}/dc{dc_tag}_{file_paths["fn_error_base_actind_TYPE_SITE_summary"]}.csv')

    return activity_arr, actvtind_cont_column

In [15]:
for site_key in site_keys:
    for type_key in type_keys:
        for dc_tag in dc_tags:
            print(site_key, type_key, dc_tag)
            data_params['metric_tag'] = 'bout_time_percentage'
            data_params["site_tag"] = site_key
            data_params["site_name"] = SITE_NAMES[site_key]
            data_params["type_tag"] = type_key
            data_params["detector_tag"] = 'bd2'
            file_paths = get_file_paths(data_params)

            activitybout_arr_fn, btp_arr_fn = generate_activity_btp_for_false_negatives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=True)

Carp LF 30of30


100%|██████████| 11/11 [01:34<00:00,  8.55s/it]


Carp LF 1of6


100%|██████████| 11/11 [01:01<00:00,  5.62s/it]


Carp LF 2of6


...

100%|██████████| 11/11 [00:38<00:00,  3.51s/it]


Carp LF 10of30


100%|██████████| 11/11 [00:51<00:00,  4.71s/it]


Carp LF 15of30


100%|██████████| 11/11 [01:03<00:00,  5.78s/it]


In [16]:
for site_key in site_keys:
    for type_key in type_keys:
        for dc_tag in dc_tags:
            print(site_key, type_key, dc_tag)
            data_params['metric_tag'] = 'call_rate'
            data_params["site_tag"] = site_key
            data_params["site_name"] = SITE_NAMES[site_key]
            data_params["type_tag"] = type_key
            data_params["detector_tag"] = 'bd2'
            file_paths = get_file_paths(data_params)

            activitycallrate_arr_fn, callrate_arr_fn = generate_activity_call_rate_for_false_negatives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=True)

Carp LF 30of30


100%|██████████| 11/11 [00:25<00:00,  2.36s/it]


Carp LF 1of6


100%|██████████| 11/11 [00:35<00:00,  3.21s/it]


Carp LF 2of6


...

100%|██████████| 11/11 [00:24<00:00,  2.21s/it]


Carp LF 10of30


100%|██████████| 11/11 [00:23<00:00,  2.15s/it]


Carp LF 15of30


100%|██████████| 11/11 [00:24<00:00,  2.22s/it]


In [17]:
for site_key in site_keys:
    for type_key in type_keys:
        for dc_tag in dc_tags:
            print(site_key, type_key, dc_tag)
            data_params['metric_tag'] = 'activity_index'
            data_params["site_tag"] = site_key
            data_params["site_name"] = SITE_NAMES[site_key]
            data_params["type_tag"] = type_key
            data_params["detector_tag"] = 'bd2'
            file_paths = get_file_paths(data_params)

            activityind_arr_fn, actvtind_arr_fn = generate_activity_index_percent_for_false_negatives_investigation_for_dc_tag(data_params, file_paths, dc_tag, save=True)

Carp LF 30of30


100%|██████████| 11/11 [00:26<00:00,  2.37s/it]


Carp LF 1of6


100%|██████████| 11/11 [00:39<00:00,  3.55s/it]


Carp LF 2of6


...

100%|██████████| 11/11 [00:32<00:00,  2.91s/it]


Carp LF 10of30


100%|██████████| 11/11 [00:28<00:00,  2.63s/it]


Carp LF 15of30


100%|██████████| 11/11 [00:26<00:00,  2.41s/it]
